In [5]:
import sqlalchemy

In [6]:
sqlalchemy.__version__

'2.0.50'

In [7]:
import sqlalchemy as db

In [8]:
engine = db.create_engine('sqlite:///mydb.db')

In [9]:
conn = engine.connect()

In [10]:
metadata = db.MetaData()

In [11]:
dir(db)

['ARRAY',
 'AdaptedConnection',
 'Alias',
 'AliasedReturnsRows',
 'Any',
 'AssertionPool',
 'AsyncAdaptedQueuePool',
 'BIGINT',
 'BINARY',
 'BLANK_SCHEMA',
 'BLOB',
 'BOOLEAN',
 'BaseDDLElement',
 'BaseRow',
 'BigInteger',
 'BinaryExpression',
 'BindParameter',
 'BindTyping',
 'Boolean',
 'BooleanClauseList',
 'CHAR',
 'CLOB',
 'CTE',
 'CacheKey',
 'Case',
 'Cast',
 'CheckConstraint',
 'ChunkedIteratorResult',
 'ClauseElement',
 'ClauseList',
 'CollectionAggregate',
 'Column',
 'ColumnClause',
 'ColumnCollection',
 'ColumnDefault',
 'ColumnElement',
 'ColumnExpressionArgument',
 'ColumnOperators',
 'Compiled',
 'CompoundSelect',
 'Computed',
 'Connection',
 'Constraint',
 'CreateEnginePlugin',
 'CursorResult',
 'DATE',
 'DATETIME',
 'DDL',
 'DDLElement',
 'DECIMAL',
 'DOUBLE',
 'DOUBLE_PRECISION',
 'Date',
 'DateTime',
 'DefaultClause',
 'Delete',
 'Dialect',
 'Double',
 'Engine',
 'Enum',
 'ExceptionContext',
 'Executable',
 'ExecutableDDLElement',
 'ExecutionContext',
 'Exists',
 'Ex

In [12]:
cars = db.Table(
    'Car', metadata,
    db.Column('car_id', db.Integer, primary_key=True),
    db.Column('car_name', db.Text,),
    db.Column('car_country', db.Text),
    db.Column('car_mileage', db.Integer),
    db.Column('car_price', db.Integer)
)
cars

Table('Car', MetaData(), Column('car_id', Integer(), table=<Car>, primary_key=True, nullable=False), Column('car_name', Text(), table=<Car>), Column('car_country', Text(), table=<Car>), Column('car_mileage', Integer(), table=<Car>), Column('car_price', Integer(), table=<Car>), schema=None)

In [13]:
parking = db.Table(
    'Parking', metadata,
    db.Column('parking_id', db.Integer, primary_key=True),
    db.Column('car_id', db.Integer,),
    db.Column('parked', db.Boolean,)
)

In [14]:
metadata.create_all(engine)

In [15]:
insertion = cars.insert().values([
    {'car_name': 'Audi', 'car_country': 'Germany', 'car_mileage': 10000, 'car_price': 35500},
    {'car_name': 'BMW', 'car_country': 'Germany', 'car_mileage': 15600, 'car_price': 24500},
    {'car_name': 'Mercedes', 'car_country': 'Germany', 'car_mileage': 4000, 'car_price': 55500}

])

In [16]:
insertion2 = parking.insert().values([
    {'parking_id': 1, 'car_id': 1, 'parked': True},
    {'parking_id': 2, 'car_id': 2, 'parked': False},
    {'parking_id': 3, 'car_id': 3, 'parked': True}

])

In [17]:
conn.execute(insertion)
conn.execute(insertion2)

In [18]:
select_all_query = db.select(cars)
select_result = conn.execute(select_all_query)
select_result

In [19]:
select_result.fetchall()

[(1, 'Audi', 'Germany', 10000, 35500),
 (2, 'BMW', 'Germany', 15600, 24500),
 (3, 'Mercedes', 'Germany', 4000, 55500)]

In [20]:
select_all_query = db.select(cars).where(cars.columns.car_mileage > 12000)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(2, 'BMW', 'Germany', 15600, 24500)]

In [21]:
#ORDER_BY

select_all_query = db.select(cars).order_by(cars.columns.car_price)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(2, 'BMW', 'Germany', 15600, 24500),
 (1, 'Audi', 'Germany', 10000, 35500),
 (3, 'Mercedes', 'Germany', 4000, 55500)]

In [22]:
#GROUP_BY

select_all_query = db.select(cars.columns.car_name, cars.columns.car_mileage).group_by(cars.columns.car_name)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[('Audi', 10000), ('BMW', 15600), ('Mercedes', 4000)]

In [23]:
#JOIN

select_all_query = db.select(cars, parking).join(parking, cars.columns.car_id == parking.columns.car_id)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(1, 'Audi', 'Germany', 10000, 35500, 1, 1, True),
 (2, 'BMW', 'Germany', 15600, 24500, 2, 2, False),
 (3, 'Mercedes', 'Germany', 4000, 55500, 3, 3, True)]

In [24]:
#PODZAPROSY

select_all_query = db.select(cars).where(cars.columns.car_price > (db.select (db.func.avg(cars.columns.car_price)).scalar_subquery()))
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(3, 'Mercedes', 'Germany', 4000, 55500)]

In [25]:
insertion = cars.insert().values([
    {'car_name': 'Lada', 'car_country': 'Russia', 'car_mileage': 1000, 'car_price': 500}

])

In [26]:
conn.execute(insertion)

In [27]:
select_all_query = db.select(cars)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(1, 'Audi', 'Germany', 10000, 35500),
 (2, 'BMW', 'Germany', 15600, 24500),
 (3, 'Mercedes', 'Germany', 4000, 55500),
 (4, 'Lada', 'Russia', 1000, 500)]

In [28]:
update_query = db.update(cars).where(cars.columns.car_name == 'Lada').values(car_price=0)

In [29]:
conn.execute(update_query)

In [30]:
select_all_query = db.select(cars)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(1, 'Audi', 'Germany', 10000, 35500),
 (2, 'BMW', 'Germany', 15600, 24500),
 (3, 'Mercedes', 'Germany', 4000, 55500),
 (4, 'Lada', 'Russia', 1000, 0)]

In [31]:
delete_query = db.delete(cars).where(cars.columns.car_name == 'Lada')
conn.execute(delete_query)

In [32]:
select_all_query = db.select(cars)
select_result = conn.execute(select_all_query)
select_result.fetchall()

[(1, 'Audi', 'Germany', 10000, 35500),
 (2, 'BMW', 'Germany', 15600, 24500),
 (3, 'Mercedes', 'Germany', 4000, 55500)]

In [33]:
# with
#   idx as (
#         select distinct id
#         from A
#       )
# select *
# from A where id in ((
#         select distinct id
#         from A
#       ))
# join B  where id in (
#     (
#         select distinct id
#         from A
#       )
# )
# on
#   )